# PAA KX2

The [PAA KX2](https://www.peakanalysisandautomation.com/products/plate-handlers/kx2/) (also sold as KiNEDx KX2) is a 5-axis SCARA plate-handling robot from Peak Analysis and Automation. It supports:

- [Arms](../../capabilities/arms) (pick/place, Cartesian and joint movement, freedrive teaching)

The device communicates over CAN bus using the CANopen protocol (CiA-301 + CiA-402 drive profile). A PCAN/SocketCAN-compatible CAN interface is required (e.g. PEAK PCAN-USB).

| Model | PLR Name | Gripper | Notes |
|---|---|---|---|
| KX2 | `KX2` | servo | 4 motion axes (shoulder, Z, elbow, wrist) + servo gripper |

## Setup

`setup()` opens the CAN bus, discovers nodes, reads drive parameters, enables the motion axes, and homes + closes the servo gripper.

```{note}
This notebook uses `KX2Canopen`, the [`canopen`](https://pypi.org/project/canopen/)-library-backed driver. The legacy hand-rolled transport is still available as `KX2`; both expose the same arm-capability frontend (`kx2.arm`), so switching between them is a one-line import change.
```

In [ ]:
from pylabrobot.paa.kx2 import KX2Canopen

kx2 = KX2Canopen()
await kx2.setup()

## Arm capabilities

The KX2 exposes an {class}`~pylabrobot.capabilities.arms.orientable_arm.OrientableArm` on `kx2.arm`, backed by {class}`~pylabrobot.paa.kx2.KX2ArmBackend`. Gripper yaw is controlled via a single `direction` float (degrees; 0° = front). For the full arm API, see [Arms](../../capabilities/arms).

The raw driver (`kx2.driver`, a `KX2Driver`) stays available for low-level access — motor commands, drive diagnostics, binary interpreter, etc.

### Gripper control

The servo gripper is force-aware. `close_gripper` moves to the target width and then verifies the plate is gripped; pass `check_plate_gripped=False` via `GripParams` to skip the verification move. Units are KX2 gripper encoder units (mm-equivalent once calibrated).

In [2]:
await kx2.arm.open_gripper(gripper_width=30)

cmd {<KX2Axis.SERVO_GRIPPER: 6>: 30}


In [3]:
from pylabrobot.paa.kx2 import KX2ArmBackend

await kx2.arm.close_gripper(
    gripper_width=0,
    backend_params=KX2ArmBackend.GripParams(check_plate_gripped=False),
)

cmd {<KX2Axis.SERVO_GRIPPER: 6>: 0}


In [4]:
await kx2.arm.is_gripper_closed()

True

Adjust the default gripping force (as a percentage of the motor's peak current) before closing on fragile labware:

In [5]:
await kx2.arm.backend.servo_gripper_set_default_gripping_force(max_force_percent=30)

node_id not int: KX2Axis.SERVO_GRIPPER <enum 'KX2Axis'>
motor send command 6 PL 1 0.9200000166893005 True (2)
node_id not int: KX2Axis.SERVO_GRIPPER <enum 'KX2Axis'>
motor send command 6 CL 1 0.2730000078678131 True (2)
node_id not int: KX2Axis.SERVO_GRIPPER <enum 'KX2Axis'>
motor send command 6 PL 1 0.27600000500679017 True (2)


### Cartesian movement

Move the arm to a Cartesian location. `direction` is the gripper yaw in degrees (Z-axis rotation only — the KX2 cannot roll or pitch). Use {class}`~pylabrobot.paa.kx2.KX2ArmBackend.CartesianMoveParams` to override velocity and acceleration.

In [6]:
await kx2.arm.request_gripper_location()

GripperLocation(location=Coordinate(x=349.124, y=170.552, z=280.0441), rotation=Rotation(x=0, y=0, z=198.93898581596522))

In [11]:
from pylabrobot.resources import Coordinate

await kx2.arm.move_to_location(
    location=Coordinate(x=340.124, y=200.552, z=290.0441),
    direction=198,
    backend_params=KX2ArmBackend.CartesianMoveParams(vel_pct=25, accel_pct=25),
)

cmd {<KX2Axis.SHOULDER: 1>: -59.474537393375996, <KX2Axis.Z: 2>: 290.0441, <KX2Axis.ELBOW: 3>: 243.19263109523325, <KX2Axis.WRIST: 4>: 257.474537393376}


### Joint movement

Move in joint space using the {class}`~pylabrobot.paa.kx2.KX2Axis` enum. Rotary axes in degrees; Z (and gripper) in mm-equivalent encoder units.

```{warning}
Moving to arbitrary joint positions can cause the arm to collide with its base, gripper, or nearby equipment. Verify coordinates in a safe workspace first, and start with low velocity.
```

In [ ]:
from pylabrobot.paa.kx2 import KX2Axis

position = {
    KX2Axis.SHOULDER: 0.0,
    KX2Axis.Z: 150.0,
    KX2Axis.ELBOW: 90.0,
    KX2Axis.WRIST: 0.0,
}
await kx2.arm.backend.move_to_joint_position(
    position,
    backend_params=KX2ArmBackend.JointMoveParams(vel_pct=25, accel_pct=25),
)

### Querying position

Read the current joint angles or Cartesian end-effector pose:

In [12]:
await kx2.arm.backend.request_joint_position()

{<KX2Axis.SHOULDER: 1>: 300.52548395979784,
 <KX2Axis.Z: 2>: 290.0440412368887,
 <KX2Axis.ELBOW: 3>: 243.19265215891733,
 <KX2Axis.WRIST: 4>: 257.4755859375,
 <KX2Axis.SERVO_GRIPPER: 6>: 0.0}

In [13]:
await kx2.arm.request_gripper_location()

GripperLocation(location=Coordinate(x=340.1239, y=200.5521, z=290.044), rotation=Rotation(x=0, y=0, z=198.00106989729784))

### Pick and place

`pick_up_at_location` moves to the target pose and closes the gripper to `resource_width`. `drop_at_location` moves and opens the gripper. Approach and retract motions are the caller's responsibility — bracket these calls with your own pre- and post-moves.

In [ ]:
pick  = Coordinate(x=450.0, y=0.0,    z=80.0)
place = Coordinate(x=450.0, y=-200.0, z=80.0)
above = Coordinate(x=0,     y=0,      z=60.0)
yaw   = 0.0

await kx2.arm.move_to_location(pick + above, direction=yaw)
await kx2.arm.pick_up_at_location(location=pick, direction=yaw, resource_width=0)
await kx2.arm.move_to_location(pick + above, direction=yaw)

await kx2.arm.move_to_location(place + above, direction=yaw)
await kx2.arm.drop_at_location(location=place, direction=yaw, resource_width=30)
await kx2.arm.move_to_location(place + above, direction=yaw)

### Freedrive (teaching mode)

Freedrive disables torque on the motion axes so you can push the arm by hand; the servo gripper stays powered. The KX2 frees all motion axes at once — the `free_axes` argument is accepted for API parity and ignored.

In [ ]:
await kx2.arm.backend.start_freedrive_mode(free_axes=[0])

In [ ]:
# Manually guide the arm to the desired pose, then read it:
taught = await kx2.arm.request_gripper_location()
print(taught)

In [ ]:
await kx2.arm.backend.stop_freedrive_mode()

### Halt and fault diagnostics

`halt` issues an emergency stop on every motion axis. Driver-level methods give you estop-line state and per-axis fault codes.

In [ ]:
await kx2.arm.backend.halt()

In [14]:
await kx2.arm.backend.get_estop_state()

node_id not int: KX2Axis.SHOULDER <enum 'KX2Axis'>
motor send command 1 SR 1  False (1)
!!! not the same


True

In [ ]:
await kx2.driver.motor_get_fault(KX2Axis.SHOULDER)

## Teardown

In [ ]:
await kx2.stop()